# Problem 59: Knowledge Conflict Resolution

**Difficulty:** Hard | **Framework:** LangChain

**Categories:** rag, prompt-design

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/knowledge-conflict-resolution).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must detect when retrieved documents contain contradictory information
- Contradictions must be flagged explicitly in the response
- The agent must not silently pick one version over another
- Both conflicting sources must be cited when a conflict is detected


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="The company's annual revenue for 2024 was $150M, a 20% increase.", metadata={"source": "investor_report_q4.pdf"}),
    Document(page_content="Total 2024 revenue reached $142M according to the audited financial statements.", metadata={"source": "audited_financials_2024.pdf"}),
    Document(page_content="The company has 500 employees across 5 global offices.", metadata={"source": "company_overview.pdf"}),
    Document(page_content="Current headcount stands at 480 full-time employees.", metadata={"source": "hr_report_dec2024.pdf"}),
]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on context.\n\nContext: {context}"),
    ("human", "{question}"),
])

# BUG: Picks the first document's answer with no conflict detection
def ask(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    chain = prompt | llm
    result = chain.invoke({"context": context, "question": question})
    return result.content

# The two revenue docs contradict — the agent should flag this
print(ask("What was the company's 2024 revenue?"))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. After retrieval, add a step that compares the retrieved documents for conflicting claims
2. Use an LLM to detect whether two documents agree or contradict on the same topic
3. When a conflict is found, present both versions with their sources and let the user decide


## 7. Evaluation Criteria

Check your solution against these criteria:

- Detects contradictions
- Flags conflicts explicitly
- Cites both conflicting sources
- Handles agreeing documents normally
